# 3 — Modeling & Evaluation
Notebook ini memuat `data/processed/dataset_clean.csv`, split train/test, scaling, training model, evaluasi, dan menyimpan model terbaik.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import DataPreprocessor
from src.model import ModelBuilder
from src.evaluation import ModelEvaluator

PROCESSED_PATH = ROOT / 'data' / 'processed' / 'dataset_clean.csv'
MODEL_OUT = ROOT / 'results' / 'models' / 'best_model.pkl'
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH

WindowsPath('C:/Users/bobok/modul2_datamining/data/processed/dataset_clean.csv')

In [2]:
# Load processed data
if not PROCESSED_PATH.exists():
    raise FileNotFoundError('dataset_clean.csv belum ada. Jalankan notebook 2_preprocessing dulu.')

df = pd.read_csv(PROCESSED_PATH)
print('Shape:', df.shape)
df.head()

Shape: (25, 6)


,age,income,education_level,years_experience,monthly_spending,purchased
0,22,3200,3,1,500,0
1,25,4500,0,2,650,0
2,28,5200,1,4,700,0
3,30,6100,1,5,850,1
4,33,7200,1,7,920,1


In [3]:
# Prepare features & target
preferred_targets = ['purchased', 'loan_approved', 'target', 'label']
target_col = next((c for c in preferred_targets if c in df.columns), None)
if target_col is None:
    target_col = df.columns[-1]
    print('⚠️ Target column tidak ditemukan di daftar default; memakai kolom terakhir:', target_col)
else:
    print('✓ Target column:', target_col)

X = df.drop(columns=[target_col])
y = df[target_col]

X.shape, y.shape

✓ Target column: purchased


((25, 5), (25,))

In [4]:
# Split + scaling (fit on train, transform on test)
builder = ModelBuilder()
X_train, X_test, y_train, y_test = builder.split_data(X, y, test_size=0.2)

scaler = DataPreprocessor()
X_train_s = scaler.normalize_scale(X_train, X_train.columns, method='standard', fit=True)
X_test_s = scaler.normalize_scale(X_test, X_test.columns, method='standard', fit=False)

X_train_s.head()

✓ Data split selesai
  Train set: 20 samples
  Test set: 5 samples
✓ Data di-scale menggunakan Standard Scaler
✓ Data di-scale menggunakan Standard Scaler


,age,income,education_level,years_experience,monthly_spending
9,1.502611,1.480881,0.75,1.595672,1.585362
13,-0.445218,-0.457877,-0.50,-0.398918,-0.323435
1,-1.280002,-1.200380,-1.75,-1.285402,-1.065745
22,-0.306087,-0.375377,-0.50,-0.398918,-0.288087
5,0.111305,0.243376,0.75,0.044324,0.100742


In [5]:
# Train models
model_lr = builder.train_logistic_regression(X_train_s, y_train)
model_rf = builder.train_random_forest(X_train_s, y_train, n_estimators=200)

✓ Logistic Regression model dilatih
✓ Random Forest model dilatih (200 trees)


In [6]:
# Evaluate
evaluator = ModelEvaluator()

y_pred_lr = model_lr.predict(X_test_s)
y_pred_rf = model_rf.predict(X_test_s)

print('Logistic Regression:')
m_lr = evaluator.evaluate_classification(y_test, y_pred_lr)

print('Random Forest:')
m_rf = evaluator.evaluate_classification(y_test, y_pred_rf)

comparison = evaluator.compare_models({'LogReg': model_lr, 'RandomForest': model_rf}, X_test_s, y_test)
comparison

Logistic Regression:
CLASSIFICATION METRICS
accuracy       : 1.0000
precision      : 1.0000
recall         : 1.0000
f1             : 1.0000
Random Forest:
CLASSIFICATION METRICS
accuracy       : 1.0000
precision      : 1.0000
recall         : 1.0000
f1             : 1.0000

MODEL COMPARISON
       Model  Accuracy
      LogReg       1.0
RandomForest       1.0


,Model,Accuracy
0,LogReg,1.0
1,RandomForest,1.0


In [7]:
# Save best model
best_model = model_rf if m_rf.get('accuracy', 0) >= m_lr.get('accuracy', 0) else model_lr
builder.save_model(best_model, str(MODEL_OUT))
MODEL_OUT

✓ Model disimpan ke: C:\Users\bobok\modul2_datamining\results\models\best_model.pkl


WindowsPath('C:/Users/bobok/modul2_datamining/results/models/best_model.pkl')

## Selesai
- Model terbaik tersimpan di `results/models/best_model.pkl`
- Anda bisa lanjut melakukan tuning dengan `ModelBuilder.hyperparameter_tuning()` jika diperlukan.